# Customer Retention Analytics

**Business question:** Which customer and account characteristics are associated with churn, and how accurately can churn risk be predicted?

**Tools:** Python, Pandas, DuckDB SQL, scikit-learn, Matplotlib

## 1. Load and inspect the data

The path logic supports either running this notebook from the project folder or from its `Python/` directory.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

candidates = [Path('data/WA_Fn-UseC_-Telco-Customer-Churn.csv'), Path('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')]
DATA_PATH = next(p for p in candidates if p.exists())
df = pd.read_csv(DATA_PATH)
df.shape, df.head()

In [ ]:
df.info()
df.describe(include='all').T

## 2. Clean the data

`TotalCharges` contains values that need numeric conversion. Rows that remain missing after conversion are removed.

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna().copy()
df['Churn_Flag'] = df['Churn'].map({'Yes': 1, 'No': 0})
print(df.isna().sum().sort_values(ascending=False).head())
print(f'Overall churn rate: {df["Churn_Flag"].mean():.1%}')

## 3. Exploratory business analysis

In [ ]:
contract_churn = (df.groupby('Contract')['Churn_Flag'].mean().sort_values(ascending=False) * 100)
contract_churn

In [ ]:
plt.figure(figsize=(7,4.5))
contract_churn.plot(kind='bar')
plt.ylabel('Churn rate (%)')
plt.title('Customer Churn Rate by Contract')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
df.groupby('Churn')[['tenure','MonthlyCharges']].mean()

### Interpretation

Use the tables and charts above to describe differences in churn, tenure, and monthly charges. Avoid treating these group differences as causal effects.

## 4. SQL analysis

In [ ]:
import duckdb
con = duckdb.connect()
con.register('customers', df)
con.sql('''
SELECT Contract,
       COUNT(*) AS customers,
       AVG(Churn_Flag) AS churn_rate
FROM customers
GROUP BY Contract
ORDER BY churn_rate DESC
''').df()

The standalone version of these queries is in `SQL/customer_retention_analysis.sql`.

## 5. Churn prediction model

A logistic regression model estimates churn probability. Numeric variables are standardized and categorical variables are one-hot encoded.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

numeric = ['tenure','MonthlyCharges','TotalCharges','SeniorCitizen']
categorical = ['Partner','Dependents','InternetService','Contract','PaymentMethod']
X = df[numeric + categorical]
y = df['Churn_Flag']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
preprocess = ColumnTransformer([
    ('num', StandardScaler(), numeric),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical)
])
model = Pipeline([('preprocess', preprocess), ('model', LogisticRegression(max_iter=1000))])
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]
print(classification_report(y_test, y_pred))
print('ROC-AUC:', round(roc_auc_score(y_test, y_prob), 3))

## 6. Limitations

- Churn associations do not establish causation.
- The model is a portfolio project, not a production retention system.
- A probability threshold should be selected using business costs for false positives and false negatives.
- The dataset is historical and may not represent a current customer base.